# GHSL — built-up surface

Download GHS-BUILT-S 2020 at 100 m over the Lisbon AOI: the built-up surface (m² per cell).

## Setup

The imports for the whole notebook. `pyramids` provides `Dataset` (reading the GeoTIFF); `earthlens` provides the unified `EarthLens` entry point. `tempfile` gives us a throwaway output directory for the download.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from earthlens.core import EarthLens
from pyramids.dataset import Dataset

## Download GHS-BUILT-S

We build the request first — source `ghsl`, the `GHS_BUILT_S` variable, the 2020 epoch, and a bounding box over Lisbon. `path` points at a temporary directory so the downloaded GeoTIFF lands somewhere disposable.

In [ ]:
out = tempfile.mkdtemp()
builtup = EarthLens(
    data_source='ghsl',
    variables=['GHS_BUILT_S'],
    start='2020-01-01',
    end='2020-12-31',
    aoi=[-9.20, 38.68, -9.10, 38.78],
    path=out,
)

GHSL is a file-writing backend, so `download()` returns the list of written output paths. The download step is kept separate from construction so each is easy to read and re-run.

In [ ]:
paths = builtup.download(progress_bar=False)
paths

## Read and clean the raster

Read the first (only) written file into a `Dataset`, pull out its array, and mask the GHSL nodata fill (negative values) to `NaN` so it plots cleanly.

In [ ]:
arr = np.asarray(Dataset.read_file(str(paths[0])).read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
arr = np.where(arr < 0, np.nan, arr)

## Plot the built-up surface

Render the masked array: each cell holds the built-up surface in m², so denser urban cores show up brightest on the `cividis` colour map.

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(arr, cmap='cividis')
plt.colorbar(label='built m2 / cell')
plt.title('GHS-BUILT-S 2020 (100 m) — Lisbon')
plt.axis('off')
plt.show()